In [0]:
print(dbutils)

In [0]:
from pyspark.sql.functions import current_timestamp


catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
table = dbutils.widgets.get('table')

bucket = dbutils.widgets.get('bucket')

source = f"{bucket}/raw/{table}"
table_name = f"{catalog}.{schema}.{table}"
SchemaLocation = f"{bucket}/_checkpoints/{schema}/{table}/schemaLocation/"
CheckpointLocation = f"{bucket}/_checkpoints/{schema}/{table}/checkpoints/"

df = spark.readStream.format("CloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("SchemaEvolutionMode","AddNewColumns")\
    .option("InferSchema", "true")\
    .option("cloudFiles.schemaLocation", SchemaLocation)\
    .load(source)
    

df.withColumn("date", current_timestamp())\
    .writeStream\
    .outputMode("append")\
    .option("mergeSchema", "true")\
    .option("checkpointLocation", CheckpointLocation)\
    .trigger(availableNow=True).toTable(table_name)